In [27]:
# Core scverse libraries
import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt





# === INPUT PATHS ===

rna_input_file = '/home/ajarrah/PhD_Thesis/chapter_3/aggregated_h5ad_data/aggregated_mouse_brain_202502.h5ad'
rna_input_folder = "/home/ajarrah/PhD_Thesis/chapter_3/h5ad_data/"
rna_tissue_positions_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_4/tissue_positions/tissue_positions.csv"

gene_image_output_dir = "/home/ajarrah/PhD_Thesis/chapter_4/images/genes/"

rna_spot_spacing = 100  # µm center-to-center

In [28]:
rna_adata = sc.read_h5ad(rna_input_file)

rna_aad_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_aad_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_aad_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_aad_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_ac_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_ac_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_ac_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_ac_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_yad_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yad_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yad_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yad_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yc_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Young_Control_Mouse_Brain_202502.h5ad"))
rna_yc_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Young_Control_Mouse_Brain_202502.h5ad"))
rna_yc_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Young_Control_Mouse_Brain_202502.h5ad"))
rna_yc_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Young_Control_Mouse_Brain_202502.h5ad"))

rna_data_name = [rna_yc_1, rna_yc_2, rna_yc_3, rna_yc_4,
                rna_yad_1, rna_yad_2, rna_yad_3, rna_yad_4,
                rna_ac_1, rna_ac_2, rna_ac_3, rna_ac_4,
                rna_aad_1, rna_aad_2, rna_aad_3, rna_aad_4]

rna_sample_name = ["YC_1", "YC_2", "YC_3", "YC_4",
                "YAD_1", "YAD_2", "YAD_3", "YAD_4",
                "AC_1", "AC_2", "AC_3", "AC_4",
                "AAD_1", "AAD_2", "AAD_3", "AAD_4"]

rna_sample_display_name = ["Young Control 1", "Young Control 2", "Young Control 3", "Young Control 4",
                       "Young AD 1", "Young AD 2", "Young AD 3", "Young AD 4",
                       "Aged Control 1", "Aged Control 2", "Aged Control 3", "Aged Control 4",
                       "Aged AD 1", "Aged AD 2", "Aged AD 3", "Aged AD 4"]

# Ensure matrix is dense or use .A1 for sparse matrices
if isinstance(rna_adata.X, np.ndarray):
    non_zero_genes = rna_adata.X.sum(axis=0) != 0
else:
    non_zero_genes = np.array((rna_adata.X.sum(axis=0) != 0)).ravel()  # for sparse matrix

rna_adata = rna_adata[:, non_zero_genes].copy()


# Normalize total counts per cell (CPM / library size normalization)
sc.pp.normalize_total(rna_adata, target_sum=1e2)   # or 1e6 depending on preference
for data in rna_data_name:
    sc.pp.normalize_total(data, target_sum=1e2)   # or 1e6 depending on preference

# Read the CSV
# Read only selected columns
tissue_positions = pd.read_csv(
    rna_tissue_positions_csv_input_path,
    usecols=["barcode", "array_row", "array_col"])

# Show first rows
print(tissue_positions.head())


rna_spot_spacing = 100  # µm center-to-center
n_rows = tissue_positions['array_row'].max() + 1
n_cols = tissue_positions['array_col'].max() + 1

coords_um = []
for _, row in tissue_positions.iterrows():
    r, c = row['array_row'], row['array_col']
    y = r * (np.sqrt(3)/2 * rna_spot_spacing)
    x = c * rna_spot_spacing/2 
    coords_um.append((x, y))

coords_um = np.array(coords_um)
tissue_positions['x_um'] = coords_um[:, 0]
tissue_positions['y_um'] = coords_um[:, 1]

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


              barcode  array_row  array_col
0  ACGCCTGACACGCGCT-1          0          0
1  TACCGATCCAACACTT-1          1          1
2  ATTAAAGCGGACGAGC-1          0          2
3  GATAAGGGACGATTAG-1          1          3
4  GTGCAAATCACCAATA-1          0          4


In [29]:
from matplotlib.colors import LinearSegmentedColormap

# === COLOR SCALE ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),      # dark gray
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),     # navy
    (0.15, "#0000FF"),     # blue
    (0.30, "#8000FF"),     # purple-ish
    (0.45, "#FF0000"),     # red
    (0.60, "#FF8000"),     # orange
    (0.75, "#FFFF00"),     # yellow
    (1.0, "#FFFFFF")       # white
])


In [30]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Circle, Polygon
from scipy.spatial import cKDTree

def rna_plot_producer_interpolater(
    adata,
    tissue_positions,
    gene,
    rna_sample_name="YC_1",
    output_dir="plots",
    spot_diameter=55,
    cmap_name="viridis",
    figsize=(12, 12),
    alpha_rhombi=0.7,
    draw_poly_edges=False,
    draw_spot_edges=True,
    show_colorbar=True,
    show_axis=True,
    show_title=True,
    background_color="#00BF00"
):
    """
    Plot spatial transcriptomics spots with interpolated rhombus regions
    and save the figure as an image.

    Parameters
    ----------
    adata : AnnData
        Expression data (must include `var_names` for genes).
    tissue_positions : pd.DataFrame
        DataFrame with columns ['barcode', 'x_um', 'y_um'].
    gene : str
        Gene to visualize.
    rna_sample_name : str
        Name of the RNA sample for naming the file.
    output_dir : str
        Base directory to save images.
    show_colorbar : bool
        Whether to include the colorbar.
    show_axis : bool
        Whether to show axis lines, labels, and ticks.
    show_title : bool
        Whether to display a title.
    """

    # --- Step 1: extract data ---
    spot_radius = spot_diameter / 2
    x = tissue_positions["x_um"].values
    y = tissue_positions["y_um"].values

    expr = adata[:, adata.var_names == gene].X.toarray().flatten()
    barcode_to_expr = dict(zip(adata.obs_names, expr))
    expr_full = tissue_positions["barcode"].map(barcode_to_expr).to_numpy()

    vmin, vmax = np.nanmin(expr_full), np.nanmax(expr_full)

    # --- Step 2: build KDTree ---
    coords = np.column_stack([x, y])
    tree = cKDTree(coords)

    # --- Step 3: compute rhombi ---
    mid_polys, mid_vals = [], []
    for i, (xi, yi, vi) in enumerate(zip(x, y, expr_full)):
        if np.isnan(vi):
            continue
        dists, idxs = tree.query([xi, yi], k=7)  # self + 6 neighbors
        for j in idxs[1:]:
            if np.isnan(expr_full[j]):
                continue
            xj, yj, vj = x[j], y[j], expr_full[j]

            # Midpoint and averaged value
            mx, my = (xi + xj) / 2, (yi + yj) / 2
            mv = (vi + vj) / 2

            dx, dy = xj - xi, yj - yi
            length = np.sqrt(dx**2 + dy**2)
            if length == 0:
                continue

            px, py = -dy / length, dx / length
            half_short = length / (2 * np.sqrt(3))

            p1 = (mx - dx / 2, my - dy / 2)
            p2 = (mx + dx / 2, my + dy / 2)
            p3 = (mx + px * half_short, my + py * half_short)
            p4 = (mx - px * half_short, my - py * half_short)

            mid_polys.append([p1, p3, p2, p4])
            mid_vals.append(mv)

    # --- Step 4: plot ---
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(background_color)
    ax.set_aspect("equal")

    cmap = plt.get_cmap(cmap_name) if isinstance(cmap_name, str) else cmap_name
    norm = Normalize(vmin=vmin, vmax=vmax)

    # rhombi
    for poly, val in zip(mid_polys, mid_vals):
        patch = Polygon(
            poly,
            closed=True,
            facecolor=cmap(norm(val)),
            edgecolor="k" if draw_poly_edges else "none",
            lw=0.2,
            alpha=alpha_rhombi,
        )
        ax.add_patch(patch)

    # circles
    for xi, yi, val in zip(x, y, expr_full):
        if not np.isnan(val):
            circle = Circle(
                (xi, yi),
                radius=spot_radius,
                facecolor=cmap(norm(val)),
                edgecolor="k" if draw_spot_edges else "none",
                lw=0.2,
            )
            ax.add_patch(circle)

    ax.set_xlim(x.min() - 2 * spot_radius, x.max() + 2 * spot_radius)
    ax.set_ylim(y.min() - 2 * spot_radius, y.max() + 2 * spot_radius)
    ax.invert_yaxis()

    # Optional components
    if show_axis:
        ax.set_xlabel("x (µm)")
        ax.set_ylabel("y (µm)")
    else:
        ax.axis("off")

    if show_title:
        ax.set_title(f"{gene} expression (spots + interpolated rhombi)")

    if show_colorbar:
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        fig.colorbar(sm, ax=ax, label=f"{gene} expression")

    # --- Step 5: save figure ---
    sample_dir = os.path.join(output_dir, rna_sample_name)
    os.makedirs(sample_dir, exist_ok=True)
    filename = f'{gene}|{rna_sample_name}|{vmin:.2f}|{vmax:.2f}.png'
    fig.savefig(os.path.join(sample_dir, filename), dpi=300, bbox_inches='tight', transparent=False)
    plt.close(fig)

In [31]:
rna_plot_producer_interpolater(
    adata = rna_data_name[0],
    tissue_positions=tissue_positions,
    gene="Apoe",
    rna_sample_name=rna_sample_name[0],
    output_dir=gene_image_output_dir,
    spot_diameter=55,
    cmap_name=custom_cmap,
    figsize=(10, 10),
    alpha_rhombi=1,
    draw_poly_edges=False,
    draw_spot_edges=False,
    show_colorbar=False,
    show_axis=False,
    show_title=False,
    background_color="#00BF00"
)

In [36]:
import numpy as np
import scanpy as sc
import os

# --- Step 1: count how many samples each gene is lowly expressed in ---
all_genes = rna_data_name[0].var_names
gene_low_counts = {gene: 0 for gene in all_genes}

for adata in rna_data_name:
    spot_counts = np.array((adata.X > 0).sum(axis=0)).ravel()
    low_expr_mask = spot_counts <= 500
    low_genes = adata.var_names[low_expr_mask]
    for gene in low_genes:
        gene_low_counts[gene] += 1

# --- Step 2: identify genes lowly expressed (≤10 spots) in ≥15 samples ---
low_expr_genes_across_15 = [gene for gene, count in gene_low_counts.items() if count >= 8]

print(f"Number of genes expressed in ≤10 spots in ≥15 samples: {len(low_expr_genes_across_15)}")

# --- Step 3: filter them out ---
for i, adata in enumerate(rna_data_name):
    genes_to_keep = ~adata.var_names.isin(low_expr_genes_across_15)
    filtered = adata[:, genes_to_keep].copy()
    rna_data_name[i] = filtered
    print(f"{rna_sample_name[i]}: filtered -> {filtered.n_vars} genes remain")

# --- Step 4 (optional): save filtered files ---
# for adata, name in zip(rna_data_name, rna_sample_name):
#     out_path = os.path.join(rna_input_folder, f"{name}_filtered.h5ad")
#     adata.write(out_path)
#     print(f"Saved: {out_path}")


Number of genes expressed in ≤10 spots in ≥15 samples: 355
YC_1: filtered -> 7297 genes remain
YC_2: filtered -> 7297 genes remain
YC_3: filtered -> 7297 genes remain
YC_4: filtered -> 7297 genes remain
YAD_1: filtered -> 7297 genes remain
YAD_2: filtered -> 7297 genes remain
YAD_3: filtered -> 7297 genes remain
YAD_4: filtered -> 7297 genes remain
AC_1: filtered -> 7297 genes remain
AC_2: filtered -> 7297 genes remain
AC_3: filtered -> 7297 genes remain
AC_4: filtered -> 7297 genes remain
AAD_1: filtered -> 7297 genes remain
AAD_2: filtered -> 7297 genes remain
AAD_3: filtered -> 7297 genes remain
AAD_4: filtered -> 7297 genes remain


In [ ]:
import numpy as np

# Step 1: get feature names from the first sample
all_genes = rna_data_name[0].var_names

# Step 2: initialize a set of genes expressed in <3 spots in all samples
low_expr_genes_across_all = set(all_genes)

# Step 3: iterate over all samples
for adata in rna_data_name:
    # Count number of nonzero spots (cells) per gene
    spot_counts = np.array((adata.X > 0).sum(axis=0)).ravel()
    
    # Identify genes expressed in fewer than 3 spots for this sample
    low_expr_genes = set(adata.var_names[spot_counts < 590])
    
    # Keep only genes that are lowly expressed in all samples so far
    low_expr_genes_across_all &= low_expr_genes

# Step 4: convert result to sorted list
low_expr_genes_across_all = sorted(low_expr_genes_across_all)

print(f"Number of genes expressed in <3 spots in ALL samples: {len(low_expr_genes_across_all)}")
print("Example genes:", low_expr_genes_across_all[:20])


In [ ]:
for i in range(15):
    rna_plot_producer_interpolater(
        adata = rna_data_name[i],
        tissue_positions=tissue_positions,
        gene="Amot",
        rna_sample_name=rna_sample_name[i],
        output_dir=gene_image_output_dir,
        spot_diameter=55,
        cmap_name=custom_cmap,
        figsize=(10, 10),
        alpha_rhombi=1,
        draw_poly_edges=False,
        draw_spot_edges=False,
        show_colorbar=False,
        show_axis=False,
        show_title=False,
        background_color="#00BF00"
    )